# Extraction de données — FakeNewsNet

Ce notebook part des CSV bruts FakeNewsNet (`id`, `news_url`, `title`, `tweet_ids`, un fichier par label) pour récupérer le texte de l'article d'origine à chaque `news_url`, et écarte les entrées dont le lien n'existe plus (404 ou domaine mort). Comme pour Google Fact Check, seul le texte est collecté ici — pas d'image.

## Sommaire
1. Configuration
2. Lecture d'un fichier source
3. Récupération de l'article (texte + statut HTTP)
4. Exécution de la collecte
5. Filtrage des liens morts
6. Sauvegarde par fichier
7. Exploration rapide

## 1. Configuration

In [1]:
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

In [7]:
PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "FakeNewsNet"
OUTPUT_DIR = PROJECT_ROOT / "data" / "fakenewsnet" / "raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HTTP_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; Projet12-FactCheckCollector/1.0)"}
REQUEST_DELAY_SECONDS = 1.0

# FakeNewsNet ships one file per label - map each to the canonical label used by the pipeline.
SOURCE_FILES = [
    ("gossipcop_fake.csv", "fake"),
    ("gossipcop_real.csv", "true"),
    ("politifact_fake.csv", "fake"),
    ("politifact_real.csv", "true"),
]

# Full files have thousands of rows: cap per file to keep runtime reasonable (1 req/s).
# Set to None to process a whole file.
MAX_ARTICLES_PER_FILE = 400

# Statuses that mean "this link no longer exists", beyond an outright connection failure.
DEAD_LINK_STATUSES = {404, 410}

## 2. Lecture d'un fichier source

`news_url` est parfois sans schéma (ex. `www.dailymail.co.uk/...`) : on le normalise avant toute requête.

In [3]:
def normalize_url(url) -> str | None:
    """Add a scheme if missing; return None for empty/missing input."""
    if pd.isna(url):
        return None
    candidate = str(url).strip()
    if not candidate:
        return None
    if not re.match(r"^[a-zA-Z][a-zA-Z0-9+.\-]*://", candidate):
        candidate = f"https://{candidate}"
    return candidate

In [4]:
def read_fakenewsnet_csv(filename: str, label: str, max_rows: int | None = None) -> pd.DataFrame:
    """Read one raw FakeNewsNet CSV and tag it with its label.

    Rows are randomly sampled (not just the first max_rows) so the subset isn't
    biased toward the oldest entries at the top of the file, which tend to be the
    most likely to have dead links anyway.
    """
    df = pd.read_csv(RAW_DATA_DIR / filename)
    df["label"] = label
    if max_rows is not None and len(df) > max_rows:
        df = df.sample(n=max_rows, random_state=42).reset_index(drop=True)
    return df

## 3. Récupération de l'article (texte + statut HTTP)

Une seule requête par URL : on garde le code de statut (pour le filtrage des liens morts) et on extrait le texte principal de la page si elle répond correctement.

In [5]:
def fetch_article(url, min_length: int = 200) -> tuple[int | None, str | None]:
    """Fetch a news article page. Returns (status_code, article_text).

    status_code is None on a network-level failure (timeout, DNS failure, ...) -
    as dead as a 404 in practice, but not an HTTP status per se. article_text is
    None whenever the page isn't a usable 200, or its extracted text is too short.
    """
    normalized_url = normalize_url(url)
    if normalized_url is None:
        return None, None

    try:
        response = requests.get(normalized_url, headers=HTTP_HEADERS, timeout=15, allow_redirects=True)
    except requests.RequestException:
        return None, None

    if response.status_code != 200:
        return response.status_code, None

    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    article = soup.find("article") or soup.find("main") or soup.find(attrs={"role": "main"})
    paragraphs = (article or soup).find_all("p")

    text = "\n".join(p.get_text(" ", strip=True) for p in paragraphs)
    text = re.sub(r"\n{2,}", "\n", text).strip()

    return response.status_code, (text if len(text) >= min_length else None)

## 4. Exécution de la collecte

Une passe par fichier source ; chaque `df` accumule `http_status` et `article_text`.

In [8]:
results: dict[str, pd.DataFrame] = {}

for filename, label in SOURCE_FILES:
    df = read_fakenewsnet_csv(filename, label, max_rows=MAX_ARTICLES_PER_FILE)

    statuses, texts = [], []
    for url in tqdm(df["news_url"], desc=filename):
        status, text = fetch_article(url)
        statuses.append(status)
        texts.append(text)
        time.sleep(REQUEST_DELAY_SECONDS)

    df["http_status"] = statuses
    df["article_text"] = texts
    results[filename] = df
    print(f"{filename}: fetched {len(df)} row(s)")

gossipcop_fake.csv:   0%|          | 0/400 [00:00<?, ?it/s]

C:\Users\johan\AppData\Local\Temp\ipykernel_528\1713559805.py:20: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(response.text, "html.parser")


gossipcop_fake.csv: fetched 400 row(s)


gossipcop_real.csv:   0%|          | 0/400 [00:00<?, ?it/s]

gossipcop_real.csv: fetched 400 row(s)


politifact_fake.csv:   0%|          | 0/400 [00:00<?, ?it/s]

politifact_fake.csv: fetched 400 row(s)


politifact_real.csv:   0%|          | 0/400 [00:00<?, ?it/s]

politifact_real.csv: fetched 400 row(s)


## 5. Filtrage des liens morts

On regarde d'abord la distribution des statuts HTTP rencontrés, puis on écarte les 404/410 et les échecs de connexion (statut manquant).

In [9]:
for filename, df in results.items():
    print(f"--- {filename} ---")
    print(df["http_status"].value_counts(dropna=False).sort_index())

--- gossipcop_fake.csv ---
http_status
200.0    223
202.0     12
400.0      2
402.0      1
403.0     85
404.0     31
406.0      5
410.0      2
418.0      1
503.0      1
NaN       37
Name: count, dtype: int64
--- gossipcop_real.csv ---
http_status
200.0    259
401.0      1
403.0     64
404.0     35
406.0      2
429.0      1
500.0      3
503.0      1
NaN       34
Name: count, dtype: int64
--- politifact_fake.csv ---
http_status
200.0    188
202.0      2
400.0      1
403.0     39
404.0     48
406.0      1
410.0      4
429.0      1
444.0      2
500.0      1
503.0      4
530.0      1
NaN      108
Name: count, dtype: int64
--- politifact_real.csv ---
http_status
200.0    235
400.0     10
401.0      2
403.0     51
404.0     35
406.0      3
410.0      4
500.0      1
NaN       59
Name: count, dtype: int64


In [10]:
for filename in list(results):
    df = results[filename]
    before = len(df)

    is_dead_link = df["http_status"].isna() | df["http_status"].isin(DEAD_LINK_STATUSES)
    df = df[~is_dead_link].reset_index(drop=True)

    results[filename] = df
    print(f"{filename}: dropped {before - len(df)}/{before} dead link(s), kept {len(df)}")

gossipcop_fake.csv: dropped 70/400 dead link(s), kept 330
gossipcop_real.csv: dropped 69/400 dead link(s), kept 331
politifact_fake.csv: dropped 160/400 dead link(s), kept 240
politifact_real.csv: dropped 98/400 dead link(s), kept 302


## 5b. Nettoyage des fichiers sources

Écraser les CSV sources originaux avec les données filtrées (colonnes originales uniquement : `id`, `news_url`, `title`, `tweet_ids`). La prochaine exécution ne retraitera pas les liens déjà identifiés comme morts.

In [11]:
for filename, label in SOURCE_FILES:
    df = results[filename]
    
    # Keep only the original columns (drop http_status and article_text).
    original_columns = ["id", "news_url", "title", "tweet_ids"]
    df_cleaned = df[original_columns].copy()
    
    source_path = RAW_DATA_DIR / filename
    df_cleaned.to_csv(source_path, index=False)
    print(f"Cleaned {source_path}: {len(df_cleaned)} row(s)")

Cleaned c:\Formation\Projet_12\data\FakeNewsNet\gossipcop_fake.csv: 330 row(s)
Cleaned c:\Formation\Projet_12\data\FakeNewsNet\gossipcop_real.csv: 331 row(s)
Cleaned c:\Formation\Projet_12\data\FakeNewsNet\politifact_fake.csv: 240 row(s)
Cleaned c:\Formation\Projet_12\data\FakeNewsNet\politifact_real.csv: 302 row(s)


## 6. Sauvegarde par fichier

Un CSV par fichier source, prêt pour le pipeline de transformation :
`uv run python -m projet_12.pipeline.run --source fakenewsnet --input data/fakenewsnet/raw/<fichier> --output data/processed/<fichier> --label <fake|true>`

(l'adaptateur `fakenewsnet` doit être réactivé dans `src/projet_12/pipeline/adapters/__init__.py` s'il est commenté).

In [12]:
for filename, df in results.items():
    output_path = OUTPUT_DIR / filename
    df.to_csv(output_path, index=False)
    print(f"Saved {len(df)} row(s) to {output_path}")

Saved 330 row(s) to c:\Formation\Projet_12\data\fakenewsnet\raw\gossipcop_fake.csv
Saved 331 row(s) to c:\Formation\Projet_12\data\fakenewsnet\raw\gossipcop_real.csv
Saved 240 row(s) to c:\Formation\Projet_12\data\fakenewsnet\raw\politifact_fake.csv
Saved 302 row(s) to c:\Formation\Projet_12\data\fakenewsnet\raw\politifact_real.csv


## 7. Exploration rapide

In [13]:
summary = pd.DataFrame(
    [
        {
            "file": filename,
            "label": df["label"].iloc[0] if len(df) else None,
            "rows_kept": len(df),
            "rows_with_article_text": df["article_text"].notna().sum(),
        }
        for filename, df in results.items()
    ]
)
summary

,file,label,rows_kept,rows_with_article_text
0,gossipcop_fake.csv,fake,330,190
1,gossipcop_real.csv,true,331,228
2,politifact_fake.csv,fake,240,132
3,politifact_real.csv,true,302,160


In [15]:
sample_df = next(iter(results.values()))
sample_df[sample_df["article_text"].notna()][["title", "label", "http_status", "article_text"]].head(100)

,title,label,http_status,article_text
3,Brad Pitt and Jennifer Aniston 'busted' over A...,fake,200.0,Sign Up to the Newsletter Sign up to receive t...
4,Ellen DeGeneres and Portia de Rossi 'going to ...,fake,200.0,Ellen DeGeneres and Portia de Rossi's relation...
5,Tom Cruise 'hasn't seen his daughter in four y...,fake,200.0,Tom Cruise has reportedly not seen his daughte...
6,Ben Affleck's History With Alcohol Addiction: ...,fake,200.0,Ben Affleck has returned to rehab as he contin...
8,Queen Elizabeth bans pasta dishes from Bucking...,fake,200.0,Sign Up to the Newsletter Sign up to receive t...
...,...,...,...,...
162,Star Wars' Mark Hamill mocks his own death hoa...,fake,200.0,It's not the first time he's faced his own dea...
163,Who Will Be Meghan Markle's Maid of Honor?,fake,200.0,The answer is not what you expect.\nBack in 20...
164,Angelina Jolie at risk of losing primary custo...,fake,200.0,See more of our coverage in your search result...
168,Kate Middleton: Is She Pregnant With Twin Girls?!,fake,200.0,"By Author Tyler Johnson\nPublished: Feb 15, 20..."
